# LOGISTPULSE · OLAP en memoria sobre el Data Lake

Capa **Hadoop / Machine Learning** del diagrama: aquí consultamos los Parquet de la zona *curated* con DuckDB (OLAP en memoria) y cruzamos con el DW y las reseñas de redes sociales (zona *raw*).

In [ ]:
import duckdb, pandas as pd, matplotlib.pyplot as plt
from lp.common import query_df, get_bytes, RAW

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("""SET s3_endpoint='minio:9000'; SET s3_use_ssl=false; SET s3_url_style='path';
           SET s3_access_key_id='minioadmin'; SET s3_secret_access_key='minioadmin123';""")
con.sql("SELECT COUNT(*) AS filas FROM 's3://lake-curated/shipments/shipments.parquet'").show()

## 1. Cubo OLAP en memoria: OTIF por mes y transportista

In [ ]:
df = con.sql("""
  SELECT month, carrier,
         ROUND(100.0*SUM(CASE WHEN on_time THEN 1 ELSE 0 END)/COUNT(*),1) AS otif_pct
  FROM 's3://lake-curated/shipments/shipments.parquet'
  WHERE status='DELIVERED'
  GROUP BY CUBE(month, carrier) ORDER BY month, carrier
""").df()
df.dropna().pivot(index='month', columns='carrier', values='otif_pct').plot(figsize=(9,4), title='OTIF por mes y transportista')
plt.ylabel('% a tiempo'); plt.show()

## 2. Lluvia (dato de terceros) vs atraso

In [ ]:
rain = con.sql("""
  SELECT CAST(FLOOR(rain_mm/5)*5 AS INT) AS lluvia_mm,
         ROUND(100.0*AVG(CASE WHEN on_time THEN 0 ELSE 1 END),1) AS pct_tarde, COUNT(*) AS n
  FROM 's3://lake-curated/shipments/shipments.parquet'
  WHERE status='DELIVERED' AND rain_mm IS NOT NULL GROUP BY 1 ORDER BY 1
""").df()
rain.plot.bar(x='lluvia_mm', y='pct_tarde', legend=False, figsize=(8,3.5), title='% de entregas tarde según lluvia (mm)'); plt.show()

## 3. Redes sociales: reseñas (no estructuradas) vs puntualidad
Las reseñas viven solo en el Data Lake (*raw*); se unen al DW por `order_id`.

In [ ]:
import io, json
rev = pd.DataFrame([json.loads(l) for l in get_bytes(RAW, 'social/reviews.jsonl').decode().splitlines() if l])
ship = query_df("SELECT order_id, on_time FROM dw.fact_shipment WHERE status='DELIVERED'")
m = rev.merge(ship, on='order_id')
m.groupby('on_time')['rating'].agg(['mean','count']).round(2)